01. Setup

In [1]:
# Instalação do Java e Spark
!apt-get update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark

# Variáveis de ambiente
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

# Pacote Iceberg
!wget -q https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.6.1/iceberg-spark-runtime-3.5_2.12-1.6.1.jar -P /content/spark-3.5.0-bin-hadoop3/jars/

import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col

# Sessão Spark
spark = SparkSession.builder \
.appName("IcebergWithSpark") \
.config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
.config("spark.sql.catalog.hadoop_catalog", "org.apache.iceberg.spark.SparkCatalog") \
.config("spark.sql.catalog.hadoop_catalog.type", "hadoop") \
.config("spark.sql.catalog.hadoop_catalog.warehouse", "/content/warehouse") \
.config("spark.sql.default.catalog", "hadoop_catalog") \
.getOrCreate()

# Diretório do Warehouse
!mkdir -p /content/warehouse

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,951 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,945 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,011 kB]
Get:14 http://security.ubunt

# 02. Snapshots e TimeTravel

In [2]:
from datetime import datetime, timedelta

In [3]:
# Criamos a tabela vendas usando Iceberg
spark.sql("""
    CREATE TABLE IF NOT EXISTS hadoop_catalog.default.vendas (
        id INT,
        produto STRING,
        quantidade INT,
        preco DOUBLE,
        data_venda DATE
    )
    USING iceberg
""")


DataFrame[]

In [4]:
# Dados Iniciais
data_initial = [
    (1, "Produto A", 10, 15.5, "2023-11-01"),
    (2, "Produto B", 5, 22.0, "2023-11-02"),
    (3, "Produto C", 8, 30.0, "2023-11-03")
]

columns = ["id", "produto", "quantidade", "preco", "data_venda"]

df_initial = spark.createDataFrame(data_initial, columns)
df_initial = df_initial.withColumn("data_venda", to_date(col("data_venda"), "yyyy-MM-dd"))

# Gravamos os dados
df_initial.writeTo("hadoop_catalog.default.vendas").append()

In [5]:
# Visualização dos dados
spark.sql("SELECT * FROM hadoop_catalog.default.vendas").show()

+---+---------+----------+-----+----------+
| id|  produto|quantidade|preco|data_venda|
+---+---------+----------+-----+----------+
|  1|Produto A|        10| 15.5|2023-11-01|
|  2|Produto B|         5| 22.0|2023-11-02|
|  3|Produto C|         8| 30.0|2023-11-03|
+---+---------+----------+-----+----------+



In [6]:
# Lista de snapshots atuais da tabela vendas
snapshots_df = spark.sql("SELECT * FROM hadoop_catalog.default.vendas.snapshots")
snapshots_df.select("snapshot_id", "committed_at", "operation").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|6434044036620655723|2026-03-30 20:20:34.819|append   |
+-------------------+-----------------------+---------+



In [7]:
# Incluimos mais dados
data_additional = [
    (4, "Produto D", 12, 25.0, "2023-11-04"),
    (5, "Produto E", 7, 18.5, "2023-11-05")
]

df_additional = spark.createDataFrame(data_additional, columns)
df_additional = df_additional.withColumn("data_venda", to_date(col("data_venda"), "yyyy-MM-dd"))
df_additional.writeTo("hadoop_catalog.default.vendas").append()

In [8]:
# Visualização dos dados
spark.sql("SELECT * FROM hadoop_catalog.default.vendas order by id").show()

+---+---------+----------+-----+----------+
| id|  produto|quantidade|preco|data_venda|
+---+---------+----------+-----+----------+
|  1|Produto A|        10| 15.5|2023-11-01|
|  2|Produto B|         5| 22.0|2023-11-02|
|  3|Produto C|         8| 30.0|2023-11-03|
|  4|Produto D|        12| 25.0|2023-11-04|
|  5|Produto E|         7| 18.5|2023-11-05|
+---+---------+----------+-----+----------+



In [9]:
# List os snapshots novamente
snapshots_df = spark.sql("SELECT * FROM hadoop_catalog.default.vendas.snapshots")
snapshots_df.select("snapshot_id", "committed_at", "operation").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|6434044036620655723|2026-03-30 20:20:34.819|append   |
|8221941293875833360|2026-03-30 20:20:41.036|append   |
+-------------------+-----------------------+---------+



In [10]:
# Atualização de dados
spark.sql("""
    UPDATE hadoop_catalog.default.vendas
    SET preco = 16.0
    WHERE id = 1
""")

DataFrame[]

In [11]:
# Visualização dos dados
spark.sql("SELECT * FROM hadoop_catalog.default.vendas  order by id").show()

+---+---------+----------+-----+----------+
| id|  produto|quantidade|preco|data_venda|
+---+---------+----------+-----+----------+
|  1|Produto A|        10| 16.0|2023-11-01|
|  2|Produto B|         5| 22.0|2023-11-02|
|  3|Produto C|         8| 30.0|2023-11-03|
|  4|Produto D|        12| 25.0|2023-11-04|
|  5|Produto E|         7| 18.5|2023-11-05|
+---+---------+----------+-----+----------+



In [12]:
# List os snapshots novamente
snapshots_df = spark.sql("SELECT * FROM hadoop_catalog.default.vendas.snapshots")
snapshots_df.select("snapshot_id", "committed_at", "operation").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|6434044036620655723|2026-03-30 20:20:34.819|append   |
|8221941293875833360|2026-03-30 20:20:41.036|append   |
|7431175737188328876|2026-03-30 20:20:44.884|overwrite|
+-------------------+-----------------------+---------+



In [13]:
# Excluímos dados
spark.sql("""
    DELETE FROM hadoop_catalog.default.vendas
    WHERE id = 2
""")

DataFrame[]

In [14]:
# Visualização dos dados
spark.sql("SELECT * FROM hadoop_catalog.default.vendas  order by id").show()

+---+---------+----------+-----+----------+
| id|  produto|quantidade|preco|data_venda|
+---+---------+----------+-----+----------+
|  1|Produto A|        10| 16.0|2023-11-01|
|  3|Produto C|         8| 30.0|2023-11-03|
|  4|Produto D|        12| 25.0|2023-11-04|
|  5|Produto E|         7| 18.5|2023-11-05|
+---+---------+----------+-----+----------+



In [15]:
# List os snapshots novamente
snapshots_df = spark.sql("SELECT * FROM hadoop_catalog.default.vendas.snapshots")
snapshots_df.select("snapshot_id", "committed_at", "operation").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|6434044036620655723|2026-03-30 20:20:34.819|append   |
|8221941293875833360|2026-03-30 20:20:41.036|append   |
|7431175737188328876|2026-03-30 20:20:44.884|overwrite|
|233119369454495060 |2026-03-30 20:20:46.61 |overwrite|
+-------------------+-----------------------+---------+



### Time Travel por ID do snapshot

In [16]:
# lista de snapshost ordenados pela data de commit
snapshot_ids = spark.sql("""
    SELECT snapshot_id
    FROM hadoop_catalog.default.vendas.snapshots
    ORDER BY committed_at ASC
""").collect()
print(snapshot_ids)

[Row(snapshot_id=6434044036620655723), Row(snapshot_id=8221941293875833360), Row(snapshot_id=7431175737188328876), Row(snapshot_id=233119369454495060)]


In [17]:
# recuperamos o estado no primeiro snapshot
first_snapshot_id = snapshot_ids[0].snapshot_id
print(f"Data at Snapshot ID {first_snapshot_id}:")
spark.read \
    .option("snapshot-id", first_snapshot_id) \
    .table("hadoop_catalog.default.vendas") \
    .show()

Data at Snapshot ID 6434044036620655723:
+---+---------+----------+-----+----------+
| id|  produto|quantidade|preco|data_venda|
+---+---------+----------+-----+----------+
|  1|Produto A|        10| 15.5|2023-11-01|
|  2|Produto B|         5| 22.0|2023-11-02|
|  3|Produto C|         8| 30.0|2023-11-03|
+---+---------+----------+-----+----------+



In [18]:
# recuperamos o estado no segundo snapshot
first_snapshot_id = snapshot_ids[1].snapshot_id
print(f"Data at Snapshot ID {first_snapshot_id}:")
spark.read \
    .option("snapshot-id", first_snapshot_id) \
    .table("hadoop_catalog.default.vendas") \
    .show()

Data at Snapshot ID 8221941293875833360:
+---+---------+----------+-----+----------+
| id|  produto|quantidade|preco|data_venda|
+---+---------+----------+-----+----------+
|  1|Produto A|        10| 15.5|2023-11-01|
|  2|Produto B|         5| 22.0|2023-11-02|
|  3|Produto C|         8| 30.0|2023-11-03|
|  4|Produto D|        12| 25.0|2023-11-04|
|  5|Produto E|         7| 18.5|2023-11-05|
+---+---------+----------+-----+----------+



### Time Travel por timestamp

In [19]:
# Recupera os timestamps de committed
snapshot_timestamps = spark.sql("""
    SELECT committed_at
    FROM hadoop_catalog.default.vendas.snapshots
    ORDER BY committed_at ASC
""").collect()
print(snapshot_timestamps)

[Row(committed_at=datetime.datetime(2026, 3, 30, 20, 20, 34, 819000)), Row(committed_at=datetime.datetime(2026, 3, 30, 20, 20, 41, 36000)), Row(committed_at=datetime.datetime(2026, 3, 30, 20, 20, 44, 884000)), Row(committed_at=datetime.datetime(2026, 3, 30, 20, 20, 46, 610000))]


In [20]:
# Verificamos o timestamp do segundo snapshot
second_snapshot_timestamp = snapshot_timestamps[1].committed_at
print(second_snapshot_timestamp)

2026-03-30 20:20:41.036000


In [21]:
# Convertemos o timestamp em milesegundos para recuperar o segundo snapshot
timestamp_ms = int(second_snapshot_timestamp.timestamp() * 1000)

print(f"Data as of {second_snapshot_timestamp}:")
spark.read \
    .option("as-of-timestamp", timestamp_ms) \
    .table("hadoop_catalog.default.vendas") \
    .show()

Data as of 2026-03-30 20:20:41.036000:
+---+---------+----------+-----+----------+
| id|  produto|quantidade|preco|data_venda|
+---+---------+----------+-----+----------+
|  4|Produto D|        12| 25.0|2023-11-04|
|  5|Produto E|         7| 18.5|2023-11-05|
|  1|Produto A|        10| 15.5|2023-11-01|
|  2|Produto B|         5| 22.0|2023-11-02|
|  3|Produto C|         8| 30.0|2023-11-03|
+---+---------+----------+-----+----------+



### Time Travel usando SQL

In [22]:
# Consulta usando ID do snapshot
print(f"Data at Snapshot ID {first_snapshot_id} using SQL:")
spark.sql(f"""
    SELECT * FROM hadoop_catalog.default.vendas
    VERSION AS OF {first_snapshot_id}
""").show()

Data at Snapshot ID 8221941293875833360 using SQL:
+---+---------+----------+-----+----------+
| id|  produto|quantidade|preco|data_venda|
+---+---------+----------+-----+----------+
|  4|Produto D|        12| 25.0|2023-11-04|
|  5|Produto E|         7| 18.5|2023-11-05|
|  1|Produto A|        10| 15.5|2023-11-01|
|  2|Produto B|         5| 22.0|2023-11-02|
|  3|Produto C|         8| 30.0|2023-11-03|
+---+---------+----------+-----+----------+



In [23]:
# Usando sql com timestamp
timestamp_str = second_snapshot_timestamp.strftime("%Y-%m-%d %H:%M:%S")
print(timestamp_str)

print(f"Data as of '{timestamp_str}' using SQL:")
spark.sql(f"""
    SELECT * FROM hadoop_catalog.default.vendas
    TIMESTAMP AS OF '{timestamp_str}'
""").show()

2026-03-30 20:20:41
Data as of '2026-03-30 20:20:41' using SQL:
+---+---------+----------+-----+----------+
| id|  produto|quantidade|preco|data_venda|
+---+---------+----------+-----+----------+
|  1|Produto A|        10| 15.5|2023-11-01|
|  2|Produto B|         5| 22.0|2023-11-02|
|  3|Produto C|         8| 30.0|2023-11-03|
+---+---------+----------+-----+----------+



### Definindo Expiração de Snapshots

In [24]:
# Definimos expiração de snapshots para mais de 1 minuto
expire_timestamp = datetime.now() - timedelta(minutes=1)
expire_timestamp_str = expire_timestamp.strftime("%Y-%m-%d %H:%M:%S")

# executa expireSnapshots
expiration_result = spark.sql(f"""
    CALL hadoop_catalog.system.expire_snapshots(
        table => 'default.vendas',
        older_than => TIMESTAMP '{expire_timestamp_str}',
        retain_last => 1
    )
""")

# Exibe os resultaods
expiration_result.show(truncate=False)

+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|deleted_data_files_count|deleted_position_delete_files_count|deleted_equality_delete_files_count|deleted_manifest_files_count|deleted_manifest_lists_count|deleted_statistics_files_count|
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|2                       |0                                  |0                                  |2                           |3                           |0                             |
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+



In [25]:
#mostra snapshots ativos
snapshots_df = spark.sql("SELECT * FROM hadoop_catalog.default.vendas.snapshots")
snapshots_df.select("snapshot_id", "committed_at", "operation").show(truncate=False)

+------------------+----------------------+---------+
|snapshot_id       |committed_at          |operation|
+------------------+----------------------+---------+
|233119369454495060|2026-03-30 20:20:46.61|overwrite|
+------------------+----------------------+---------+



In [26]:
# Exclui tabela
spark.sql("DROP TABLE hadoop_catalog.default.vendas")

DataFrame[]